In [ ]:
  pip install tensorflow opencv-python-headless easyocr pandas numpy tqdm matplotlib


In [ ]:
pip install tensorflow opencv-python-headless easyocr pandas numpy tqdm matplotlib


: 

In [ ]:
python3 -m pip install 'tensorflow[and-cuda]'
# Verify the installation:
python3 -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"

In [ ]:
!pip install easyocr
import tensorflow as tf
print("GPUs Available:", tf.config.list_physical_devices('GPU'))
from tensorflow import keras
from tensorflow.keras import layers
import cv2 # OpenCV for image processing
import easyocr # For OCR
import numpy as np
import pandas as pd # To read CSV files
import os
import math # For calculating steps per epoch

from google.colab import drive
drive.mount('/content/drive')

# --- Configuration (Mostly Same) ---
IMG_WIDTH = 224
IMG_HEIGHT = 224
IMG_CHANNELS = 3
BATCH_SIZE = 32 # Define batch size for the pipeline
EPOCHS = 10
NUM_CLASSES = 2 # 0 = Counterfeit, 1 = Authentic
MAX_TEXT_LENGTH = 50
TEXT_EMBEDDING_DIM = 16

# --- Paths (Same as before) ---
BASE_PATH = "/content/drive/MyDrive/Caro_Laptop_Files"
TRAIN_DIR = os.path.join(BASE_PATH, "train")
VALID_DIR = os.path.join(BASE_PATH, "valid")
TEST_DIR = os.path.join(BASE_PATH, "test")

TRAIN_CSV = os.path.join(TRAIN_DIR, "_classes.csv") # Or train_classes.csv etc.
VALID_CSV = os.path.join(VALID_DIR, "_classes.csv")
TEST_CSV = os.path.join(TEST_DIR, "_classes.csv")

Mounted at /content/drive


In [ ]:
# --- Auto-detect correct CSV names (Same logic as before) ---
print("Checking for CSV files...")
for path_var, dir_path in [(TRAIN_CSV, TRAIN_DIR), (VALID_CSV, VALID_DIR), (TEST_CSV, TEST_DIR)]:
    base_name = os.path.basename(dir_path) # train, valid, or test
    default_csv = os.path.join(dir_path, "_classes.csv")
    specific_csv = os.path.join(dir_path, f"{base_name}_classes.csv") # e.g., train_classes.csv

    actual_csv_path = None
    if os.path.exists(default_csv):
        actual_csv_path = default_csv
    elif os.path.exists(specific_csv):
        actual_csv_path = specific_csv

    if actual_csv_path:
        print(f"Found CSV: {actual_csv_path}")
        if path_var == TRAIN_CSV: TRAIN_CSV = actual_csv_path
        if path_var == VALID_CSV: VALID_CSV = actual_csv_path
        if path_var == TEST_CSV: TEST_CSV = actual_csv_path
    else:
         print(f"X Warning: Missing expected CSV file in {dir_path}")


Checking for CSV files...
Found CSV: /content/drive/MyDrive/Caro_Laptop_Files/train/_classes.csv
Found CSV: /content/drive/MyDrive/Caro_Laptop_Files/valid/_classes.csv
Found CSV: /content/drive/MyDrive/Caro_Laptop_Files/test/_classes.csv


In [ ]:

# --- 1. Load Metadata Only ---

def load_metadata_from_csv(csv_path, image_dir):
    """Loads only image paths and labels from the CSV."""
    image_paths = []
    labels = []

    if not os.path.exists(csv_path):
        print(f"Error: CSV file not found at {csv_path}")
        return [], []

    try:
        df = pd.read_csv(csv_path)
        df.iloc[:, 0] = df.iloc[:, 0].str.strip() # Clean filenames

        print(f"Loading metadata from {csv_path}...")
        for idx, row in df.iterrows():
            img_filename = row.iloc[0]
            label = row.iloc[1] # Assuming label is in the second column
            img_path = os.path.join(image_dir, img_filename)

            # Basic check if file exists (can refine with extension checks if needed)
            if not os.path.splitext(img_path)[1]: # If no extension add common ones
                 found = False
                 for ext in ['.jpg', '.jpeg', '.png']:
                     if os.path.exists(img_path + ext):
                         img_path = img_path + ext
                         found = True
                         break
                 if not found:
                      # print(f"Warning: Image file skipped (not found): {img_path}")
                      continue
            elif not os.path.exists(img_path):
                # print(f"Warning: Image file skipped (not found): {img_path}")
                continue

            image_paths.append(img_path)
            # Convert label (assuming 1=Authentic, others=Counterfeit=0)
            labels.append(1 if int(label) == 1 else 0)

        print(f"Loaded metadata for {len(image_paths)} images from {image_dir}")
        return image_paths, labels

    except Exception as e:
        print(f"Error reading or processing metadata from {csv_path}: {e}")
        return [], []

In [ ]:
# --- Load metadata for all sets ---
train_image_paths, train_labels = load_metadata_from_csv(TRAIN_CSV, TRAIN_DIR)
valid_image_paths, valid_labels = load_metadata_from_csv(VALID_CSV, VALID_DIR)
test_image_paths, test_labels = load_metadata_from_csv(TEST_CSV, TEST_DIR)

# Basic check if loading failed
if not train_image_paths or not valid_image_paths:
     print("\nError: Failed to load metadata for training or validation set. Exiting.")
     exit()

# --- Define Preprocessing Functions (Remain mostly the same) ---
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

Loading metadata from /content/drive/MyDrive/Caro_Laptop_Files/train/_classes.csv...
Loaded metadata for 4072 images from /content/drive/MyDrive/Caro_Laptop_Files/train
Loading metadata from /content/drive/MyDrive/Caro_Laptop_Files/valid/_classes.csv...
Loaded metadata for 122 images from /content/drive/MyDrive/Caro_Laptop_Files/valid
Loading metadata from /content/drive/MyDrive/Caro_Laptop_Files/test/_classes.csv...
Loaded metadata for 65 images from /content/drive/MyDrive/Caro_Laptop_Files/test


In [ ]:
def preprocess_image_tf(image_path):
    """Loads and preprocesses a single image - TF compatible."""
    try:
        img = cv2.imread(image_path.decode('utf-8')) # Decode path from bytes
        if img is None: return None # Handle load failure
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        img = img.astype(np.float32) / 255.0
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        return img
    except Exception as e:
        # print(f"Error processing image {image_path.decode('utf-8')}: {e}")
        return None # Return None on error

In [ ]:
# Cannot use easyocr directly inside tf.py_function easily with state (reader object).
# Option 1: Pre-calculate OCR and save to a file (recommended for speed).
# Option 2: Perform OCR *outside* the tf.data pipeline before training (slow).
# Option 3: Perform OCR inside the generator (simpler to code now, but potentially slower).
# We choose Option 3 for this example, accepting the performance trade-off.

# Initialize OCR reader globally for the generator
try:
    ocr_reader_global = easyocr.Reader(['en'], gpu=tf.test.is_gpu_available()) # Use GPU if available
    print("EasyOCR Reader initialized.")
except Exception as e:
    print(f"Error initializing EasyOCR Reader: {e}. OCR will likely fail.")
    ocr_reader_global = None

def perform_ocr_and_encode(image_path_bytes, char_map):
    """Performs OCR and encodes text - designed for the generator."""
    image_path = image_path_bytes.decode('utf-8')
    encoded_text = np.zeros((MAX_TEXT_LENGTH,), dtype=np.int32) # Default to zeros

    if ocr_reader_global is None:
        # print("OCR Reader not available.")
        return encoded_text # Return empty encoded text

    try:
        # Read text
        results = ocr_reader_global.readtext(image_path)
        extracted_text = " ".join([res[1] for res in results])
        extracted_text = extracted_text.strip().upper()

        # Truncate/Pad
        if len(extracted_text) > MAX_TEXT_LENGTH:
            extracted_text = extracted_text[:MAX_TEXT_LENGTH]

        # Encode using char_map
        for i, char in enumerate(extracted_text):
            encoded_text[i] = char_map.get(char, 0) # Use 0 for unknown/padding

    except Exception as e:
        # print(f"Error during OCR/Encoding for {image_path}: {e}")
        pass # Return zeros on error

    return encoded_text

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteEasyOCR Reader initialized.


In [ ]:
# --- Create Character Map (Only from Training Metadata) ---
# We need the character map *before* creating the generator.
# A simple way is to perform OCR on a *subset* of training images first,
# or define a fixed character set if possible.
# For now, let's build it from a sample or assume a common character set.

# --- Option A: Assume Common Characters (Simpler, less accurate) ---
# common_chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?"
# char_map = {char: i+1 for i, char in enumerate(sorted(list(common_chars)))}

# --- Option B: Build from a Sample (More robust but adds initial time) ---
print("Building character map from a sample of training data...")
sample_size = min(500, len(train_image_paths)) # Sample 500 images or fewer
sample_texts = []
temp_reader = easyocr.Reader(['en']) # Temporary reader for this step
for i in range(sample_size):
     try:
        results = temp_reader.readtext(train_image_paths[i])
        sample_texts.append(" ".join([res[1] for res in results]).strip().upper())
     except:
         continue # Ignore errors during sampling
del temp_reader # Free memory

all_chars_sampled = sorted(list(set("".join(sample_texts))))
char_map = {char: i+1 for i, char in enumerate(all_chars_sampled)} # 0 for padding/unknown
text_vocab_size = len(char_map) + 1 # +1 for the unknown/padding token 0
print(f"Character map built. Vocab size: {text_vocab_size}")

Building character map from a sample of training data...
Character map built. Vocab size: 70


In [ ]:
# --- 2. Create Data Generator Function ---

def data_generator(image_paths, labels, char_map_dict):
    """Yields batches of preprocessed images, encoded texts, and labels."""
    num_samples = len(image_paths)
    indices = np.arange(num_samples)

    # This generator yields *one sample at a time*.
    # tf.data.Dataset will handle batching.
    for i in indices:
        # Process Image
        img = preprocess_image_tf(image_paths[i].encode('utf-8')) # Pass as bytes
        if img is None: # Handle potential loading errors
             # Create a dummy image if loading fails, or skip
             # print(f"Skipping sample {i} due to image loading error.")
             img = np.zeros((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.float32)
             # continue # Or skip entirely

        # Process Text
        encoded_text = perform_ocr_and_encode(image_paths[i].encode('utf-8'), char_map_dict)

        # Process Label
        label = tf.keras.utils.to_categorical(labels[i], num_classes=NUM_CLASSES)

        yield (img, encoded_text), label # Yield tuple for (inputs, targets)

In [ ]:

# --- 3. Build tf.data.Dataset Pipelines ---

# Define output signatures for the generator
output_signature = (
    (tf.TensorSpec(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=tf.float32), # Image
     tf.TensorSpec(shape=(MAX_TEXT_LENGTH,), dtype=tf.int32)), # Text
    tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32) # Label
)

# --- Training Dataset ---
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(train_image_paths, train_labels, char_map),
    output_signature=output_signature
)
# Performance optimizations
train_dataset = train_dataset.shuffle(buffer_size=len(train_image_paths) // 10) # Shuffle data
train_dataset = train_dataset.batch(BATCH_SIZE) # Batch data
train_dataset = train_dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE) # Prefetch batches

# --- Validation Dataset ---
valid_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(valid_image_paths, valid_labels, char_map),
    output_signature=output_signature
)
valid_dataset = valid_dataset.batch(BATCH_SIZE)
valid_dataset = valid_dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# --- Test Dataset ---
if test_image_paths:
    test_dataset = tf.data.Dataset.from_generator(
        lambda: data_generator(test_image_paths, test_labels, char_map),
        output_signature=output_signature
    )
    test_dataset = test_dataset.batch(BATCH_SIZE)
    test_dataset = test_dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
else:
    test_dataset = None

print("\ntf.data pipelines created successfully.")



tf.data pipelines created successfully.


In [ ]:
# --- 4. Model Architecture (Multi-Modal - Same as before) ---
# Re-define or ensure the model architecture code from the previous step is present
def build_mm_cmds_model(num_classes, max_text_len, text_vocab_size, text_embedding_dim):
    # --- Image Input Branch ---
    image_input = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), name="image_input")
    base_model = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=image_input)
    base_model.trainable = False
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name="image_pooling")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    image_features = layers.Dense(128, activation='relu', name="image_features")(x)
    # --- Text Input Branch ---
    text_input = layers.Input(shape=(max_text_len,), name="text_input", dtype=tf.int32)
    # Use mask_zero=True in Embedding if 0 is used for padding
    t = layers.Embedding(input_dim=text_vocab_size, output_dim=text_embedding_dim, mask_zero=True, name="text_embedding")(text_input)
    #with tf.device('/CPU:0'):
    t = layers.LSTM(64, name="text_lstm")(t)
    t = layers.BatchNormalization()(t)
    t = layers.Dropout(0.3)(t)
    text_features = layers.Dense(64, activation='relu', name="text_features")(t)
    # --- Fusion ---
    fused = layers.Concatenate(name="fusion")([image_features, text_features])
    fused = layers.BatchNormalization()(fused)
    fused = layers.Dense(64, activation='relu', name="fusion_dense")(fused)
    fused = layers.Dropout(0.5)(fused)
    # --- Output Head ---
    output = layers.Dense(num_classes, activation='softmax', name="output")(fused)
    # --- Build the Model ---
    model = keras.Model(inputs=[image_input, text_input], outputs=output, name="MM_CMDS")
    return model

model = build_mm_cmds_model(
    num_classes=NUM_CLASSES,
    max_text_len=MAX_TEXT_LENGTH,
    text_vocab_size=text_vocab_size, # Use calculated vocab size
    text_embedding_dim=TEXT_EMBEDDING_DIM
)
model.summary()

Model: "MM_CMDS"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_4         │ (None, 224, 224,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_2     │ (None, 224, 224,  │          7 │ rescaling_4[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_5         │ (None, 224, 224,  │          0 │ normalization_2[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_5[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,258,181 (16.24 MB)

 Trainable params: 205,538 (802.88 KB)

 Non-trainable params: 4,052,643 (15.46 MB)

In [ ]:
# --- 5. Training with tf.data ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

# Callbacks (Same as before)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
model_checkpoint_path = os.path.join(BASE_PATH, "mm_cmds_model_tf_v2.keras") # New checkpoint name
checkpoint = tf.keras.callbacks.ModelCheckpoint(model_checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max')

# Calculate steps per epoch
steps_per_epoch = math.ceil(len(train_image_paths) / BATCH_SIZE)
validation_steps = math.ceil(len(valid_image_paths) / BATCH_SIZE)

print("\nStarting training with tf.data pipeline...")



Starting training with tf.data pipeline...


In [ ]:
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=valid_dataset,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)
print("Training finished.")

Epoch 1/10


InvalidArgumentError: Graph execution error:

Detected at node MM_CMDS_1/text_lstm_1/CudnnRNNV3 defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.11/dist-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File "/usr/local/lib/python3.11/d

In [ ]:

# --- 6. Evaluation (Using the tf.data pipeline) ---
if test_dataset:
    print("\nEvaluating on Test Set...")
    try:
        best_model = tf.keras.models.load_model(model_checkpoint_path)
        test_steps = math.ceil(len(test_image_paths) / BATCH_SIZE)
        loss, acc, prec, recall = best_model.evaluate(test_dataset, steps=test_steps)
        print(f"Test Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {recall:.4f}")

        # --- Full Metrics (Requires iterating through dataset) ---
        print("Generating detailed test metrics (may take time)...")
        y_true_classes = []
        y_pred_probs = []
        for (img_batch, txt_batch), label_batch in test_dataset:
             preds = best_model.predict_on_batch([img_batch, txt_batch])
             y_pred_probs.extend(preds)
             y_true_classes.extend(np.argmax(label_batch.numpy(), axis=1))

        y_pred_probs = np.array(y_pred_probs)
        y_pred_classes = np.argmax(y_pred_probs, axis=1)
        y_true_classes = np.array(y_true_classes)

        # (Rest of the plotting code for Classification Report, CM, ROC-AUC remains the same)
        from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
        import matplotlib.pyplot as plt
        import seaborn as sns

        class_names = ["Counterfeit (0)", "Authentic (1)"]
        print("\nClassification Report:")
        # Ensure y_true and y_pred have the same length before reporting
        min_len = min(len(y_true_classes), len(y_pred_classes))
        print(classification_report(y_true_classes[:min_len], y_pred_classes[:min_len], target_names=class_names))

        cm = confusion_matrix(y_true_classes[:min_len], y_pred_classes[:min_len])
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.title("Confusion Matrix")
        plt.show()

        if NUM_CLASSES == 2 and len(y_pred_probs) > 0:
            fpr, tpr, _ = roc_curve(y_true_classes[:min_len], y_pred_probs[:min_len, 1]) # Prob of positive class
            roc_auc = auc(fpr, tpr)
            plt.figure(figsize=(6, 5))
            plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
            plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title('ROC-AUC Curve')
            plt.legend(loc="lower right")
            plt.show()

    except Exception as e:
        print(f"Could not load model or evaluate test set: {e}")
else:
    print("\nSkipping test set evaluation as no test data was found.")




In [ ]:
# --- 7. Inference Function (Needs slight adaptation) ---
# Keep the inference function similar, but ensure preprocessing matches
# The character map 'char_map' is needed for inference.
# Make sure ocr_reader_global is initialized if not already.

def predict_medicine_mm_tfdata(image_path, model, char_map_dict):
    """Predicts using the multi-modal model (tf.data style preprocessing)."""
    # 1. Preprocess Image (using the same function)
    img = preprocess_image_tf(image_path.encode('utf-8')) # Pass bytes
    if img is None: return {"error": "Could not process image"}
    img_batch = np.expand_dims(img, axis=0) # Add batch dimension

    # 2. Perform OCR and preprocess text (using the same function)
    encoded_text = perform_ocr_and_encode(image_path.encode('utf-8'), char_map_dict)
    text_batch = np.expand_dims(encoded_text, axis=0) # Add batch dimension

    # 3. Make Prediction
    prediction = model.predict([img_batch, text_batch])
    predicted_class_index = np.argmax(prediction[0])
    confidence = prediction[0][predicted_class_index]

    # 4. Decode Label
    predicted_label = "Authentic (1)" if predicted_class_index == 1 else "Counterfeit (0)"

    return {
        "image_path": image_path,
        # "extracted_text": text.strip(), # Need to re-run OCR or return encoded version
        "predicted_label": predicted_label,
        "confidence": float(confidence),
        "raw_probabilities": prediction[0].tolist()
    }

# Example Inference (Load best model first)
try:
    inference_model = tf.keras.models.load_model(model_checkpoint_path)
    print(f"\nLoaded model from {model_checkpoint_path} for inference.")

    test_image_example = os.path.join(TEST_DIR, "your_test_image_name.jpg") # *** REPLACE FILENAME ***
    if os.path.exists(test_image_example):
         result = predict_medicine_mm_tfdata(test_image_example, inference_model, char_map)
         print("\n--- Inference Result ---")
         import json
         print(json.dumps(result, indent=2))
    else:
         print(f"Test image not found: {test_image_example}")

except Exception as e:
    print(f"Could not load model for inference: {e}")


print("\nCode execution finished.")